### [이미지 폴더 -> CSV 변환]

- 데이터 파일들 : 
    * animals/train/cat
    * animals/train/dog
    * animals/test/cat
    * animals/test/dog

- 이미지들을 읽어서, CSV 파일로 저장
    * 1행 = 1개 이미지
    * 첫 번째 컬럼: label (cat=0, dog=1)
    * 나머지 컬럼: pixel0, pixel1, ... (흑백 64x64 = 4096개, 컬러 64x64 = 12288개)

- 결과물:
    - 흑백 : animals_train_gray64.csv, animals_test_gray64.csv
    - 컬러 : animals_train_rgb64.csv, animals_test_rgb64.csv

>> **[1] 모듈 로딩 및 설정**

In [1]:
## -----------------------------
## 모듈로딩
## -----------------------------

## 파일 및 폴더 관련 모듈
import glob

## CSV 파일 처리 관련 모듈
import csv

## 이미지 및 데이터 관련 모듈
from PIL import Image
import numpy as np
import pandas as pd

import os

In [2]:
## -----------------------------
## 설정
## -----------------------------
## => 폴더
BASE_DIR = "./animals"                  # 압축 해제된 animals 폴더 경로 (본인 환경에 맞게 수정)
OUT_DIR  = "../Data/animals"            # csv 저장 경로 (본인 환경에 맞게 수정)

## => 이미지 크기
SIZE = 64                               # 리사이즈 크기 (64x64)

## => 이미지 분류 라벨
LABEL_MAP = {"cat": 0, "dog": 1}    
NAME_MAP = {v:k for k, v in LABEL_MAP.items()}

## => 이미지 확장자
EXTS = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp", "*.gif", "*.JPG", "*.JPEG", "*.PNG")


>> **[2] 이미지 데이터 수집관련 함수들**

In [3]:
## -------------------------------------------------------------
## 함수이름 : collect_files
## 함수기능 : 폴더 안의 이미지 파일 목록을 반환
## 매개변수 : split - train, test 구분 폴더명
##           cls - cat, dog 구분할 라벨/클래스/타겟 이름
## 함수결과 : 정렬된 파일리스트 반환
## -------------------------------------------------------------
def collect_files(split: str, cls: str) -> list:
    # 파일리스트 저장 변수
    files = []

    # 해당 확장자 파일만 추출
    for ext in EXTS:
        files += glob.glob(f"{BASE_DIR}/{split}/{cls}/{ext}")
    return sorted(set(files))


In [4]:
## -------------------------------------------------------------
## 함수이름 : load_image_array
## 함수기능 : 이미지를 열어서 mode('L'=흑백, 'RGB'=컬러) 변환 후 리사이즈
## 매개변수 : path - 이미지 파일 경로
##           mode - 'L', 'RGB' 흑백과 컬러지정
## 함수결과 : 1차원 이미지 픽셀값 반환
## -------------------------------------------------------------
def load_image_array(path: str, mode: str) -> np.ndarray:
    ## 이미지 로딩 및 컬러스페이스변환
    img = Image.open(path).convert(mode)

    ## 크기 변환
    img = img.resize((SIZE, SIZE), Image.LANCZOS)

    ## ndarray로 변환
    arr = np.array(img, dtype=np.uint8)

    ## ndarray 2D -> 1D
    return arr.flatten()


In [5]:
## -------------------------------------------------------------
## 함수이름 : build_csv
## 함수기능 : 데이터를 읽어 csv 파일로 저장
## 함수결과 : 이미지 데이터 저장한 csv 파일 생성됨
## -------------------------------------------------------------
def build_csv(split: str, mode: str, out_path: str):
    n_channels = 1 if mode == "L" else 3
    header = ["label"] + [f"pixel{i}" for i in range(SIZE * SIZE * n_channels)]

    rows = []
    skipped = []
    for cls in ["cat", "dog"]:
        for f in collect_files(split, cls):
            try:
                arr = load_image_array(f, mode)
                rows.append([LABEL_MAP[cls]] + arr.tolist())
            except Exception as e:
                skipped.append((f, str(e)))

    with open(out_path, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(header)
        writer.writerows(rows)

    print(f"[{split} / {mode}] {len(rows)}개 저장 완료 -> {out_path}")
    if skipped:
        print(f"  스킵된 파일 {len(skipped)}개: {skipped}")


>> **[3]이미지데이터 파일 생성**

In [6]:
## ------------------------------------------------------------
## [3-1] 흑백 64x64 버전 csv 파일
## ------------------------------------------------------------
## 폴더 생성
os.makedirs(OUT_DIR, exist_ok=True)

## 파일 생성
build_csv("train", "L", f"{OUT_DIR}/animals_train_gray64.csv")
build_csv("test", "L", f"{OUT_DIR}/animals_test_gray64.csv")


c:\Users\Win11Pro\miniconda3\envs\DL_PY311\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[train / L] 229개 저장 완료 -> ../Data/animals/animals_train_gray64.csv
[test / L] 40개 저장 완료 -> ../Data/animals/animals_test_gray64.csv


In [7]:
## ------------------------------------------------------------
## [3-2] 컬러 64x64 버전 csv 파일
## ------------------------------------------------------------
build_csv("train", "RGB", f"{OUT_DIR}/animals_train_rgb64.csv")
build_csv("test", "RGB", f"{OUT_DIR}/animals_test_rgb64.csv")


[train / RGB] 229개 저장 완료 -> ../Data/animals/animals_train_rgb64.csv
[test / RGB] 40개 저장 완료 -> ../Data/animals/animals_test_rgb64.csv


In [8]:
## ------------------------------------------------------------
## 생성된 CSV 파일 확인
## ------------------------------------------------------------
df_train_gray = pd.read_csv(f"{OUT_DIR}/animals_train_gray64.csv")
df_test_gray = pd.read_csv(f"{OUT_DIR}/animals_test_gray64.csv")
df_train_rgb = pd.read_csv(f"{OUT_DIR}/animals_train_rgb64.csv")
df_test_rgb = pd.read_csv(f"{OUT_DIR}/animals_test_rgb64.csv")

print("train_gray64:", df_train_gray.shape, dict(df_train_gray["label"].value_counts()))
print("test_gray64 :", df_test_gray.shape, dict(df_test_gray["label"].value_counts()))
print("train_rgb64 :", df_train_rgb.shape, dict(df_train_rgb["label"].value_counts()))
print("test_rgb64  :", df_test_rgb.shape, dict(df_test_rgb["label"].value_counts()))

df_train_gray.head()


train_gray64: (229, 4097) {1: np.int64(122), 0: np.int64(107)}
test_gray64 : (40, 4097) {0: np.int64(20), 1: np.int64(20)}
train_rgb64 : (229, 12289) {1: np.int64(122), 0: np.int64(107)}
test_rgb64  : (40, 12289) {0: np.int64(20), 1: np.int64(20)}


,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel4086,pixel4087,pixel4088,pixel4089,pixel4090,pixel4091,pixel4092,pixel4093,pixel4094,pixel4095
0,0,82,83,82,81,80,81,83,84,84,...,87,89,99,108,58,72,137,129,133,136
1,0,255,255,255,255,255,255,255,255,255,...,255,255,255,255,255,255,255,255,255,255
2,0,241,242,242,241,236,229,213,199,214,...,206,207,207,208,208,209,210,212,213,214
3,0,38,49,67,76,72,67,69,64,72,...,105,59,75,108,143,164,128,71,48,31
4,0,65,50,51,60,66,68,69,76,85,...,109,105,99,103,104,102,99,110,114,96
